In [ ]:
import logging
import sys
from flask import Flask, jsonify, request
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

app = Flask(__name__)

MODEL_FILE = "traffic_rf_v2.joblib"
WEATHER_CATEGORIES = [
    "Clouds",
    "Drizzle",
    "Fog",
    "Haze",
    "Mist",
    "Rain",
    "Snow",
    "Squall",
    "Thunderstorm",
]


def load_or_train_model():
    """Loads the serialized model or trains and saves a new instance if missing."""
    try:
        artifact = joblib.load(MODEL_FILE)
        logger.info(
            f"Successfully loaded serialized model from '{MODEL_FILE}'."
        )
        return artifact["model"], artifact["feature_names"]
    except FileNotFoundError:
        logger.info(
            f"Model file '{MODEL_FILE}' not found. Training model now..."
        )

        df = pd.read_csv("Metro_Interstate_Traffic_Volume.csv")
        df["date_time"] = pd.to_datetime(df["date_time"])

        df["hour"] = df["date_time"].dt.hour
        df["day_of_week"] = df["date_time"].dt.dayofweek
        df["month"] = df["date_time"].dt.month
        df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

        df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
        df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

        for w in WEATHER_CATEGORIES:
            df[f"weather_{w}"] = (df["weather_main"] == w).astype(int)

        feature_cols = [
            "temp",
            "rain_1h",
            "snow_1h",
            "clouds_all",
            "hour",
            "day_of_week",
            "month",
            "is_weekend",
            "hour_sin",
            "hour_cos",
        ] + [f"weather_{w}" for w in WEATHER_CATEGORIES]

        X = df[feature_cols]
        y = df["traffic_volume"]

        X_train, _, y_train, _ = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        model = RandomForestRegressor(
            n_estimators=50, max_depth=10, random_state=42, n_jobs=-1
        )
        model.fit(X_train, y_train)

        artifact = {"model": model, "feature_names": feature_cols}
        joblib.dump(artifact, MODEL_FILE)
        logger.info(f"Model successfully trained and saved to '{MODEL_FILE}'.")
        return model, feature_cols


# Load model into memory on server startup
model, FEATURE_NAMES = load_or_train_model()


@app.route("/", methods=["GET"])
def health_check():
    """Health check endpoint."""
    return jsonify(
        {
            "status": "healthy",
            "service": "Metro Traffic Prediction API",
            "model_version": "v2.0_RandomForest",
        }
    )


@app.route("/predict", methods=["POST"])
def predict():
    """Inference endpoint accepting JSON payload and returning traffic prediction."""
    data = request.get_json()

    if not data:
        return jsonify({"error": "Invalid or missing JSON payload"}), 400

    try:
        temp = float(data.get("temp", 288.15))
        rain_1h = float(data.get("rain_1h", 0.0))
        snow_1h = float(data.get("snow_1h", 0.0))
        clouds_all = int(data.get("clouds_all", 0))
        hour = int(data["hour"])
        day_of_week = int(data["day_of_week"])
        month = int(data.get("month", 10))
        weather_main = str(data.get("weather_main", "Clear"))

        # Derive engineered features
        is_weekend = 1 if day_of_week in [5, 6] else 0
        hour_sin = np.sin(2 * np.pi * hour / 24.0)
        hour_cos = np.cos(2 * np.pi * hour / 24.0)

        weather_encoded = {
            f"weather_{w}": (1 if weather_main == w else 0)
            for w in WEATHER_CATEGORIES
        }

        input_dict = {
            "temp": temp,
            "rain_1h": rain_1h,
            "snow_1h": snow_1h,
            "clouds_all": clouds_all,
            "hour": hour,
            "day_of_week": day_of_week,
            "month": month,
            "is_weekend": is_weekend,
            "hour_sin": hour_sin,
            "hour_cos": hour_cos,
            **weather_encoded,
        }

        input_df = pd.DataFrame([input_dict])[FEATURE_NAMES]
        prediction = float(model.predict(input_df)[0])

        if prediction <= 1193:
            congestion = "Low"
        elif prediction <= 3380:
            congestion = "Medium"
        elif prediction <= 4933:
            congestion = "High"
        else:
            congestion = "Severe"

        return jsonify(
            {
                "model_version": "v2.0_RandomForest",
                "predicted_traffic_volume": round(prediction, 2),
                "congestion_level": congestion,
                "units": "vehicles/hour",
            }
        )

    except KeyError as missing_key:
        return (
            jsonify({"error": f"Missing required input parameter: {missing_key}"}),
            400,
        )
    except Exception as e:
        return jsonify({"error": str(e)}), 500


if __name__ == "__main__":
    # Settings configured for Jupyter Notebook execution
    app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False)

2026-10-05 18:50:32,774 - INFO - Successfully loaded serialized model from 'traffic_rf_v2.joblib'.
 * Serving Flask app '__main__'
 * Debug mode: off
2026-10-05 18:50:32,816 - INFO - WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
2026-10-05 18:50:32,817 - INFO - Press CTRL+C to quit
2026-10-05 18:51:48,064 - INFO - 127.0.0.1 - - [05/Oct/2026 18:51:48] "POST /predict HTTP/1.1" 415 -


In [ ]:
import requests

# Set API Endpoint
url = "http://127.0.0.1:5000/predict"

# Sample Request Payload (Tuesday 5:00 PM Peak Commute Window)
payload = {
    "temp": 288.15,
    "rain_1h": 0.0,
    "snow_1h": 0.0,
    "clouds_all": 20,
    "hour": 17,
    "day_of_week": 1,
    "month": 10,
    "weather_main": "Clear",
}

# Execute POST request
response = requests.post(url, json=payload)

# Print Status and JSON Output
print(f"Status Code: {response.status_code}\n")
print("Response JSON:")
print(response.json())